# Evaluation on the open-source test set

Loads a trained B-net, predicts the depth profile of every test B-scan of the
open-source dataset and evaluates the predictions with the functions in
`helper_functions/Metrics_experimental_data.py`:

- `calculate_median_depth_errors`: one depth estimate per defect (median of
  the defect interior), errors over the whole test set,
- `calculate_metrics_per_simulation`: the same, for each specimen separately,
- `plot_signed_errors_by_depth`: signed error histograms per ground-truth depth.

The B-scans of one specimen are stored as consecutive files, so stacking the
predictions row by row rebuilds the depth map of each specimen
(`ROWS_PER_SIM` rows each). Run the notebook from the repository root.

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

from data.data_operators import BScanDepthDataset
from networks.Unets import BnetMean
from helper_functions.helper_functions import resize_tensor
from helper_functions.Metrics_experimental_data import (
    calculate_median_depth_errors,
    calculate_metrics_per_simulation,
    plot_signed_errors_by_depth,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Settings

In [ ]:
# Specimen and dataset properties.
THICKNESS_MM = 5.0   # specimen thickness
ROWS_PER_SIM = 174   # B-scans (image rows) per specimen

# Columns removed on the left and right of every depth profile before the
# errors are computed, so the specimen edges are not evaluated.
EDGE_CROP = 50

# Trained model (written by train_unet_bscan.py).
model_path = "open_source_dataset/trained_models/Bnet_mean/best_model_clean.pth"

# Test B-scans, depth targets and the normalisation constant of the
# training set.
test_bscans = "open_source_dataset/testing/data_bscans"
test_masks = "open_source_dataset/testing/data_masks"
normalization_path = "normalization_params_experimental_cooling_only.npz"

# cooling_frame = 0 removes no frames: the whole sequence is used, as during
# training.
cooling_phase = True
cooling_frame = 0

## Model and predictions

In [ ]:
model = BnetMean().to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

test_dataset = BScanDepthDataset(
    bscan_dir=test_bscans,
    depth_dir=test_masks,
    transform=None,
    normalization_path=normalization_path,
    cooling_phase=cooling_phase,
    cooling_frame=cooling_frame,
)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

# Predictions and ground truth stacked row by row: [number of B-scans, 512].
pred_all, mask_all = [], []
with torch.no_grad():
    for X, mask in tqdm(test_loader):
        pred_all.append(model(X.to(device)).cpu())
        mask_all.append(mask)

pred_all = torch.cat(pred_all, dim=0)
mask_all = torch.cat(mask_all, dim=0)

num_simulations = pred_all.shape[0] // ROWS_PER_SIM
print(f"{pred_all.shape[0]} B-scans, {num_simulations} specimens")

## Depth-estimation errors

Errors after removing `EDGE_CROP` columns on both sides. Each defect gets
one depth estimate, the median prediction over its interior (one edge pixel
eroded), which is compared with the true depth. Values are converted from
normalised depth to mm with the specimen thickness.

In [ ]:
# Errors over the whole test set.
pred_eval = pred_all[:, EDGE_CROP:-EDGE_CROP]
mask_eval = mask_all[:, EDGE_CROP:-EDGE_CROP]

metrics, median_pred, overlay = calculate_median_depth_errors(
    pred_eval,
    mask_eval,
    erosion_pixels=1,
    inner_percent=100,
)

print(
    f"Whole test set | MAE: {metrics['mae'] * THICKNESS_MM:.4f} mm | "
    f"MedAE: {metrics['medae'] * THICKNESS_MM:.4f} mm | "
    f"RMSE: {metrics['rmse'] * THICKNESS_MM:.4f} mm\n"
)

# The same, for every specimen separately, and their mean.
per_sim, mean_metrics = calculate_metrics_per_simulation(
    pred_eval,
    mask_eval,
    rows_per_sim=ROWS_PER_SIM,
    num_simulations=num_simulations,
    thickness_mm=THICKNESS_MM,
)

In [ ]:
# Signed error (prediction - ground truth) for every depth present in the
# test set: shows whether a depth is systematically over- or underestimated.
depth_levels = np.unique(np.round(mask_eval[mask_eval > 0].numpy(), 3))

fig, axes, depth_statistics = plot_signed_errors_by_depth(
    pred_eval,
    mask_eval,
    depth_levels=depth_levels,
    tol=1e-3,
)

## Predicted and true depth maps

For every specimen: predicted depth, ground truth and absolute error. The
`ROWS_PER_SIM` x 512 maps are resized to 512 x 512 for display only.

In [ ]:
def show_simulation(pred_all, mask_all, n, vmax=0.6):
    """Predicted depth map, ground truth and absolute error of specimen n (from 1)."""
    rows = slice((n - 1) * ROWS_PER_SIM, n * ROWS_PER_SIM)
    panels = [
        (pred_all[rows], "Predicted depth"),
        (mask_all[rows], "Ground truth"),
        (torch.abs(pred_all[rows] - mask_all[rows]), "Absolute error"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (image, title) in zip(axes, panels):
        im = ax.imshow(resize_tensor(image, 512, 512), vmin=0, vmax=vmax)
        ax.set_title(f"Specimen {n}: {title}")
        ax.set_xlabel("Pixel width")
        ax.set_ylabel("Pixel height")
    fig.colorbar(im, ax=axes, label="Normalised depth", shrink=0.8)
    plt.show()


for n in range(1, num_simulations + 1):
    show_simulation(pred_all, mask_all, n)